In [1]:
import os
import re
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from collections import Counter
from difflib import SequenceMatcher

# Optional richer interactive figures.
try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

# ============================================================
# CONFIG / PATHS
# ============================================================
# Reading from your saved Kaggle dataset
INPUT_DIR = "/kaggle/input/datasets/adithyaled24b039/phi3-stratqa-self-correction-basin-atlas-csv/strategyqa_self_correction_basin_atlas"

# Restoring to the exact original target directory
OUT_DIR = "/kaggle/working/strategyqa_self_correction_basin_atlas"

# Prime the working directory with your saved data
if os.path.exists(INPUT_DIR) and INPUT_DIR != OUT_DIR:
    print(f"Copying saved data from {INPUT_DIR} to {OUT_DIR}...")
    shutil.copytree(INPUT_DIR, OUT_DIR, dirs_exist_ok=True)

CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
HTML_DIR = os.path.join(OUT_DIR, "html")

for p in [CSV_DIR, PLOTS_DIR, REPORTS_DIR, HTML_DIR]:
    os.makedirs(p, exist_ok=True)

SEED = 42
N_RELIABILITY_BINS = 10
REPRESENTATIVE_QUESTIONS = 12
QUIVER_STEP = 2
BASELINE_VARIANT = "plain"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

# ============================================================
# UTILS
# ============================================================

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def entropy_from_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    if len(p) == 0:
        return float("nan")
    return float(-np.sum(p * np.log2(p + 1e-12)))

def preview_text(s, max_len=180):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def sequence_similarity(a, b):
    return float(SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio())

def annotated_bar(ax, fmt="{:.3f}"):
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h):
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2.0, h),
                        ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")

def bar_plot(labels, values, title, path, ylabel="Value", rotate=30, color_map="Set2"):
    plt.figure(figsize=(10.5, 4.8))
    ax = plt.gca()
    ax.bar(labels, values, color=plt.get_cmap(color_map)(np.linspace(0.12, 0.88, len(labels))))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    annotated_bar(ax)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def grouped_bar_plot(labels, v1, v2, label1, label2, title, path, ylabel="Accuracy"):
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.bar(x - width/2, v1, width, label=label1, color="tab:blue", alpha=0.85)
    ax.bar(x + width/2, v2, width, label=label2, color="tab:orange", alpha=0.85)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend(frameon=True)
    annotated_bar(ax)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def violin_plot_distributions(df, metric, group_col, title, path, ylabel="Value"):
    plt.figure(figsize=(11, 6))
    groups = df[group_col].unique()
    data = [df[df[group_col] == g][metric].dropna().to_numpy(dtype=np.float64) for g in groups]
    
    valid_idx = [i for i, d in enumerate(data) if len(d) > 0]
    valid_groups = [groups[i] for i in valid_idx]
    valid_data = [data[i] for i in valid_idx]

    if valid_data:
        parts = plt.violinplot(valid_data, showmeans=True, showmedians=False)
        for pc in parts['bodies']:
            pc.set_facecolor('tab:blue')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.xticks(range(1, len(valid_groups) + 1), valid_groups, rotation=30, ha="right")

    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def histogram(values, title, path, xlabel="Value", bins=12):
    plt.figure(figsize=(8.8, 4.4))
    plt.hist(values, bins=bins, alpha=0.88)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def scatter(x, y, title, path, xlabel="X", ylabel="Y", color=None, alpha=0.8):
    plt.figure(figsize=(7.2, 5.6))
    plt.scatter(x, y, alpha=alpha, c=color, s=26)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis", fmt=None):
    plt.figure(figsize=(max(8.5, len(xlabels) * 0.55), max(5.2, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    if fmt is not None:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if np.isfinite(mat[i, j]):
                    plt.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=7,
                             color="white" if abs(mat[i, j]) > 0.5 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def polyfit_line(x, y, deg=2):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return None, None
    deg = min(deg, len(x) - 1)
    coeffs = np.polyfit(x, y, deg=deg)
    xs = np.linspace(x.min(), x.max(), 200)
    ys = np.polyval(coeffs, xs)
    return xs, ys

# ============================================================
# SPECIFIC PLOTTING FUNCTIONS
# ============================================================

def plot_waterfall(summary_df, path):
    sdf = summary_df.sort_values("strength")
    delta = sdf["pass2_raw_accuracy"] - sdf["pass1_raw_accuracy"]
    colors = ["tab:green" if v >= 0 else "tab:red" for v in delta]
    plt.figure(figsize=(10.8, 5.0))
    plt.bar(sdf["variant"], delta, color=colors)
    plt.axhline(0, color="black", linewidth=1, alpha=0.5)
    plt.ylabel("Accuracy change (pass2 - pass1)")
    plt.title("Self-correction waterfall")
    plt.xticks(rotation=30, ha="right")
    annotated_bar(plt.gca(), fmt="{:.3f}")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def quiver_transition_plot(df, path):
    sub = df.copy()
    fig, ax = plt.subplots(figsize=(8.8, 6.4))
    color_map = {
        "correct→correct": "tab:green",
        "wrong→correct": "tab:blue",
        "correct→wrong": "tab:red",
        "wrong→wrong": "tab:gray",
    }
    colors = [color_map.get(t, "tab:gray") for t in sub["transition"].tolist()]
    ax.scatter(sub["pass1_logodds"], sub["pass1_commitment"], c=colors, alpha=0.7, s=18)
    dx = (sub["pass2_logodds"] - sub["pass1_logodds"]).to_numpy(dtype=np.float64)
    dy = (sub["pass2_commitment"] - sub["pass1_commitment"]).to_numpy(dtype=np.float64)
    ax.quiver(sub["pass1_logodds"].to_numpy(dtype=np.float64), sub["pass1_commitment"].to_numpy(dtype=np.float64), dx, dy,
              angles="xy", scale_units="xy", scale=1.0, alpha=0.35, width=0.002)
    ax.axvline(0, color="black", linewidth=1, alpha=0.35)
    ax.axhline(0.5, color="black", linewidth=1, alpha=0.2)
    ax.set_title("Self-correction trajectory field")
    ax.set_xlabel("Pass-1 log-odds")
    ax.set_ylabel("Pass-1 commitment")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def plot_strip_transitions(df, path, max_points=250):
    sub = df.sample(min(max_points, len(df)), random_state=SEED).copy() if len(df) > max_points else df.copy()
    fig, ax = plt.subplots(figsize=(12.5, 6.8))
    for i, (_, r) in enumerate(sub.iterrows()):
        color = {
            "correct→correct": "tab:green",
            "wrong→correct": "tab:blue",
            "correct→wrong": "tab:red",
            "wrong→wrong": "tab:gray",
        }.get(r["transition"], "tab:gray")
        ax.plot([0, 1], [r["pass1_logodds"], r["pass2_logodds"]], color=color, alpha=0.18, linewidth=1.0)
        ax.scatter([0, 1], [r["pass1_logodds"], r["pass2_logodds"]], color=color, alpha=0.65, s=12)
    ax.axhline(0, color="black", linewidth=1, alpha=0.35)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Pass 1", "Pass 2"])
    ax.set_ylabel("Log-odds")
    ax.set_title("Pass-wise log-odds strip plot")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def plot_response_drift(df, path):
    sub = df.copy()
    drift = sub["response_drift_sequence"].to_numpy(dtype=np.float64)
    delta = (sub["pass2_logodds"] - sub["pass1_logodds"]).to_numpy(dtype=np.float64)
    colors = sub["transition"].map({
        "correct→correct": 0,
        "wrong→correct": 1,
        "correct→wrong": 2,
        "wrong→wrong": 3,
    }).fillna(3).to_numpy(dtype=np.int64)
    plt.figure(figsize=(8.2, 6.0))
    sc = plt.scatter(drift, delta, c=colors, cmap="tab10", alpha=0.75, s=25)
    plt.axhline(0, color="black", linewidth=1, alpha=0.35)
    plt.xlabel("Explanation drift (SequenceMatcher distance)")
    plt.ylabel("Δ log-odds (pass2 - pass1)")
    plt.title("Response drift vs confidence shift")
    cbar = plt.colorbar(sc)
    cbar.set_ticks([0, 1, 2, 3])
    cbar.set_ticklabels(["cc", "wc", "cw", "ww"])
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def plot_reliability(df, variant, pass_col, path, bins=N_RELIABILITY_BINS):
    scores = np.asarray(df[f"{pass_col}_logodds"].tolist(), dtype=np.float64)
    accs = np.asarray(df[f"{pass_col}_correct"].astype(int).tolist(), dtype=np.int64)
    conf = 1.0 / (1.0 + np.exp(-scores))
    bin_edges = np.linspace(0, 1, bins + 1)
    bin_ids = np.clip(np.digitize(conf, bin_edges) - 1, 0, bins - 1)
    rows = []
    for b in range(bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            rows.append({"bin": b, "avg_conf": np.nan, "accuracy": np.nan, "count": 0})
        else:
            rows.append({
                "bin": b,
                "avg_conf": float(conf[mask].mean()),
                "accuracy": float(accs[mask].mean()),
                "count": int(mask.sum()),
            })
    r = pd.DataFrame(rows)
    r.dropna(inplace=True)
    if len(r) == 0:
        return pd.DataFrame(rows)
    plt.figure(figsize=(6.7, 5.6))
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6, label="ideal")
    plt.plot(r["avg_conf"], r["accuracy"], marker="o", linewidth=2, label=variant)
    plt.fill_between(r["avg_conf"], np.maximum(0, r["accuracy"] - 0.05), np.minimum(1, r["accuracy"] + 0.05), alpha=0.12)
    plt.xlabel("Commitment-derived confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Reliability diagram ({variant}, {pass_col})")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return r

def plot_bifurcation(df, path):
    agg = df.groupby("variant", as_index=False).agg(
        strength=("strength", "mean"),
        pass1_acc=("pass1_correct", "mean"),
        pass2_acc=("pass2_correct", "mean"),
        pass1_logodds=("pass1_logodds", "mean"),
        pass2_logodds=("pass2_logodds", "mean"),
        pass1_commit=("pass1_commitment", "mean"),
        pass2_commit=("pass2_commitment", "mean"),
    ).sort_values("strength")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    panels = [
        ("pass1_acc", "Pass-1 accuracy"),
        ("pass2_acc", "Pass-2 accuracy"),
        ("pass2_logodds", "Pass-2 log-odds"),
        ("pass2_commit", "Pass-2 commitment"),
    ]
    for ax, (col, lab) in zip(axes.ravel(), panels):
        x = agg["strength"].to_numpy(dtype=np.float64)
        y = agg[col].to_numpy(dtype=np.float64)
        ax.scatter(x, y, s=65)
        xs, ys = polyfit_line(x, y, deg=2)
        if xs is not None:
            ax.plot(xs, ys, linewidth=2)
        for _, r in agg.iterrows():
            ax.annotate(r["variant"], (r["strength"], r[col]), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Prompt strength")
        ax.set_ylabel(lab)
        ax.set_title(f"{lab} vs prompt strength")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

def plot_phase_portrait(df, path):
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.8), constrained_layout=True)
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    for ax, pass_col in zip(axes, ["pass1", "pass2"]):
        colors = df["transition"].map(cmap).tolist()
        ax.scatter(df[f"{pass_col}_logodds"], df[f"{pass_col}_commitment"], c=colors, alpha=0.72, s=20)
        ax.axvline(0, color="black", linewidth=1, alpha=0.35)
        ax.axhline(0.5, color="black", linewidth=1, alpha=0.2)
        ax.set_title(f"Phase portrait ({pass_col})")
        ax.set_xlabel("Log-odds")
        ax.set_ylabel("Commitment")
    plt.savefig(path, dpi=180)
    plt.close()

def plot_transition_heatmap(df, path):
    counts = Counter(df["transition"].tolist())
    mat = np.array([[
        counts.get("wrong→wrong", 0), counts.get("wrong→correct", 0)
    ], [
        counts.get("correct→wrong", 0), counts.get("correct→correct", 0)
    ]], dtype=np.float64)
    heatmap(mat, ["wrong→wrong", "wrong→correct"], ["wrong→wrong", "correct→wrong"], "Transition matrix (counts)", path, xlabel="Pass-2 outcome", ylabel="Pass-1 outcome", cmap="Blues", fmt="{:.0f}")

def plot_delta_heatmap(df, xcol, ycol, zcol, title, path, xbins=8, ybins=8):
    sub = df.copy()
    x = sub[xcol].to_numpy(dtype=np.float64)
    y = sub[ycol].to_numpy(dtype=np.float64)
    z = sub[zcol].astype(float).to_numpy(dtype=np.float64)
    x_edges = np.unique(np.quantile(x, np.linspace(0, 1, xbins + 1)))
    y_edges = np.unique(np.quantile(y, np.linspace(0, 1, ybins + 1)))
    
    if len(x_edges) < 3 or len(y_edges) < 3:
        return None
        
    xi = np.clip(np.digitize(x, x_edges) - 1, 0, len(x_edges) - 2)
    yi = np.clip(np.digitize(y, y_edges) - 1, 0, len(y_edges) - 2)
    mat = np.full((len(y_edges) - 1, len(x_edges) - 1), np.nan, dtype=np.float64)
    
    for i in range(len(y_edges) - 1):
        for j in range(len(x_edges) - 1):
            mask = (xi == j) & (yi == i)
            if mask.sum() > 0:
                mat[i, j] = float(z[mask].mean())
                
    # --- The Fix: Dynamically constrain bounds for TwoSlopeNorm ---
    valid_vals = mat[np.isfinite(mat)]
    if len(valid_vals) == 0:
        return None
        
    vmin = min(valid_vals.min(), 0.49)
    vmax = max(valid_vals.max(), 0.51)
    
    plt.figure(figsize=(9.4, 6.8))
    im = plt.imshow(mat, aspect="auto", origin="lower", cmap="coolwarm",
                    norm=TwoSlopeNorm(vmin=vmin, vcenter=0.5, vmax=vmax))
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return mat

def plot_mean_trajectory(df, path):
    fig, ax = plt.subplots(figsize=(8.9, 6.4))
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    for trans, g in df.groupby("transition"):
        ax.scatter(g["pass1_logodds"], g["pass1_commitment"], s=22, alpha=0.6, color=cmap.get(trans, "gray"), label=trans)
        dx = (g["pass2_logodds"] - g["pass1_logodds"]).to_numpy(dtype=np.float64)
        dy = (g["pass2_commitment"] - g["pass1_commitment"]).to_numpy(dtype=np.float64)
        ax.quiver(g["pass1_logodds"].to_numpy(dtype=np.float64)[::QUIVER_STEP], g["pass1_commitment"].to_numpy(dtype=np.float64)[::QUIVER_STEP],
                  dx[::QUIVER_STEP], dy[::QUIVER_STEP], angles="xy", scale_units="xy", scale=1.0, alpha=0.25, width=0.002)
    ax.axvline(0, color="black", linewidth=1, alpha=0.35)
    ax.set_xlabel("Pass-1 log-odds")
    ax.set_ylabel("Pass-1 commitment")
    ax.set_title("Decision trajectory field")
    ax.legend(frameon=True, fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def plot_variant_metric_dashboard(summary_df, path):
    sdf = summary_df.sort_values("strength")
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
    panels = [
        ("pass1_raw_accuracy", "Pass-1 raw accuracy"),
        ("pass2_raw_accuracy", "Pass-2 raw accuracy"),
        ("improvement_rate", "Improvement rate"),
        ("regression_rate", "Regression rate"),
        ("pass1_commitment_mean", "Pass-1 commitment"),
        ("pass2_commitment_mean", "Pass-2 commitment"),
    ]
    for ax, (metric, title) in zip(axes.ravel(), panels):
        ax.bar(sdf["variant"], sdf[metric], color=plt.get_cmap("tab10")(np.linspace(0.1, 0.9, len(sdf))))
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=30)
        annotated_bar(ax)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

def plot_question_glyph_atlas(df, q_summary, path, top_n=12):
    qdf = q_summary.sort_values(["stability_score", "transition_entropy"], ascending=[True, False]).head(top_n)
    if len(qdf) == 0:
        return
    rows = len(qdf)
    fig, axes = plt.subplots(rows, 1, figsize=(13.5, max(4.5, rows * 1.2)), constrained_layout=True)
    if rows == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, qdf.iterrows()):
        sub = df[df["question_id"] == r["question_id"]]
        counts = sub["pass2_prediction"].value_counts().reindex(["yes", "no"]).fillna(0).astype(int)
        vals = counts.values.reshape(1, -1)
        im = ax.imshow(vals, aspect="auto", cmap="magma", vmin=0, vmax=max(1, vals.max()))
        ax.set_yticks([])
        ax.set_xticks(range(2))
        ax.set_xticklabels(["yes", "no"])
        ax.set_title(f"Q{r['question_id']} | stab={r['stability_score']:.2f} | ent={r['transition_entropy']:.2f} | imp={r['improvement_rate']:.2f} | reg={r['regression_rate']:.2f} | {preview_text(r['question'], 80)}")
        for j in range(2):
            ax.text(j, 0, str(int(vals[0, j])), ha="center", va="center", color="white" if vals[0, j] > vals.max() * 0.4 else "black", fontsize=9)
    plt.suptitle("Question-level glyph atlas", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

def plot_basin_heatmaps(df, out_prefix):
    for zcol, title, fname in [
        ("improved", "Improvement basin", f"{out_prefix}_improvement_basin.png"),
        ("regressed", "Regression basin", f"{out_prefix}_regression_basin.png"),
        ("pass2_correct", "Pass-2 correctness basin", f"{out_prefix}_pass2_correct_basin.png"),
        ("pass1_vs_pass2_prediction_changed", "Prediction flip basin", f"{out_prefix}_flip_basin.png"),
    ]:
        plot_delta_heatmap(
            df,
            xcol="pass1_logodds",
            ycol="pass1_commitment",
            zcol=zcol,
            title=title,
            path=os.path.join(PLOTS_DIR, fname),
            xbins=8,
            ybins=8,
        )

def plot_representative_trajectories(df, q_summary, path):
    qdf = q_summary.copy()
    ids = []
    if len(qdf) > 0:
        ids.extend(qdf.sort_values(["stability_score", "pass1_accuracy"], ascending=[False, False]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["stability_score", "pass1_accuracy"], ascending=[True, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["improvement_rate", "stability_score"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["regression_rate", "stability_score"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["transition_entropy", "stability_score"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids = list(dict.fromkeys(ids))[:REPRESENTATIVE_QUESTIONS]
    if len(ids) == 0:
        return

    rows = len(ids)
    fig, axes = plt.subplots(rows, 2, figsize=(16, 4.4 * rows), constrained_layout=True)
    if rows == 1:
        axes = np.array([axes])
    for ridx, qid in enumerate(ids):
        sdf = df[df["question_id"] == qid].copy()
        sdf = sdf.sort_values("strength")
        ax1 = axes[ridx, 0]
        ax2 = axes[ridx, 1]
        ax1.plot(sdf["strength"], sdf["pass1_logodds"], marker="o", label="pass1", linewidth=2)
        ax1.plot(sdf["strength"], sdf["pass2_logodds"], marker="s", label="pass2", linewidth=2)
        ax1.axhline(0, color="black", linewidth=1, alpha=0.35)
        ax1.set_title(f"Q{qid} | {preview_text(sdf['question'].iloc[0], 74)}")
        ax1.set_xlabel("Prompt strength")
        ax1.set_ylabel("Log-odds")
        ax1.legend(frameon=True)

        ax2.plot(sdf["strength"], sdf["pass1_commitment"], marker="o", label="pass1", linewidth=2)
        ax2.plot(sdf["strength"], sdf["pass2_commitment"], marker="s", label="pass2", linewidth=2)
        ax2.set_title(f"Q{qid} commitment trajectory")
        ax2.set_xlabel("Prompt strength")
        ax2.set_ylabel("Commitment")
        ax2.legend(frameon=True)

    plt.suptitle("Representative stability trajectories", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

def plot_basin_map(df, path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    
    axes[0, 0].scatter(df["pass1_logodds"], df["pass1_commitment"], c=df["transition"].map(cmap), alpha=0.72, s=18)
    axes[0, 0].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[0, 0].set_title("Pass-1 basin phase portrait")
    axes[0, 0].set_xlabel("Log-odds")
    axes[0, 0].set_ylabel("Commitment")

    axes[0, 1].scatter(df["pass2_logodds"], df["pass2_commitment"], c=df["transition"].map(cmap), alpha=0.72, s=18)
    axes[0, 1].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[0, 1].set_title("Pass-2 basin phase portrait")
    axes[0, 1].set_xlabel("Log-odds")
    axes[0, 1].set_ylabel("Commitment")

    counts = Counter(df["transition"].tolist())
    mat = np.array([
        [counts.get("wrong→wrong", 0), counts.get("wrong→correct", 0)],
        [counts.get("correct→wrong", 0), counts.get("correct→correct", 0)],
    ], dtype=np.float64)
    im = axes[1, 0].imshow(mat, aspect="auto", cmap="Blues")
    axes[1, 0].set_title("Transition matrix")
    axes[1, 0].set_xticks([0, 1])
    axes[1, 0].set_xticklabels(["W→W", "W→C"])
    axes[1, 0].set_yticks([0, 1])
    axes[1, 0].set_yticklabels(["W→W", "C→W"])
    for i in range(2):
        for j in range(2):
            axes[1, 0].text(j, i, str(int(mat[i, j])), ha="center", va="center", fontsize=10)
    fig.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

    sub = df.iloc[::QUIVER_STEP].copy()
    axes[1, 1].scatter(sub["pass1_logodds"], sub["pass1_commitment"], c=sub["transition"].map(cmap), alpha=0.6, s=18)
    axes[1, 1].quiver(sub["pass1_logodds"], sub["pass1_commitment"], sub["delta_logodds"], sub["delta_commitment"],
                      angles="xy", scale_units="xy", scale=1.0, alpha=0.3, width=0.002)
    axes[1, 1].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[1, 1].axhline(0.5, color="black", linewidth=1, alpha=0.2)
    axes[1, 1].set_title("Decision vector field")
    axes[1, 1].set_xlabel("Pass-1 log-odds")
    axes[1, 1].set_ylabel("Pass-1 commitment")

    plt.suptitle("Self-correction basin atlas", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN RESCUE LOGIC
# ============================================================

def main():
    print("Loading data from restored Output Directory...")
    
    df = pd.read_csv(os.path.join(CSV_DIR, "all_results.csv"))
    variant_summary = pd.read_csv(os.path.join(CSV_DIR, "variant_summary.csv"))
    question_summary = pd.read_csv(os.path.join(CSV_DIR, "question_summary.csv"))
    
    with open(os.path.join(REPORTS_DIR, "overall_summary.json"), "r") as f:
        overall = json.load(f)

    print("Computing on-the-fly values & writing missing late-stage data...")
    best_style = variant_summary.sort_values("pass2_raw_accuracy", ascending=False).iloc[0]["variant"]

    # Re-calculate basin counts 
    basin_counts = question_summary["basin_class"].value_counts().reset_index()
    basin_counts.columns = ["basin_class", "count"]
    basin_counts.to_csv(os.path.join(CSV_DIR, "basin_class_counts.csv"), index=False)
    
    # Save the 'with calibration' CSV that was supposed to be generated right before crash
    df.to_csv(os.path.join(CSV_DIR, "all_results_with_calibration.csv"), index=False)

    print("Generating Grouped Bars & Violins...")
    grouped_bar_plot(
        variant_summary["variant"],
        variant_summary["pass1_raw_accuracy"],
        variant_summary["pass2_raw_accuracy"],
        "Pass-1", "Pass-2",
        "Pass-1 vs Pass-2 Accuracy Comparison",
        os.path.join(PLOTS_DIR, "accuracy_comparison_grouped.png")
    )

    violin_plot_distributions(df, "pass1_logodds", "variant", "Pass-1 Log-odds Distribution by Variant", os.path.join(PLOTS_DIR, "violin_pass1_logodds.png"), "Log-odds")
    violin_plot_distributions(df, "pass2_logodds", "variant", "Pass-2 Log-odds Distribution by Variant", os.path.join(PLOTS_DIR, "violin_pass2_logodds.png"), "Log-odds")
    violin_plot_distributions(df, "delta_logodds", "variant", "Delta Log-odds (Pass 2 - Pass 1) by Variant", os.path.join(PLOTS_DIR, "violin_delta_logodds.png"), "Δ Log-odds")

    print("Generating baseline & best-style dashboards...")
    plot_variant_metric_dashboard(variant_summary, os.path.join(PLOTS_DIR, "variant_metrics_dashboard.png"))
    plot_waterfall(variant_summary, os.path.join(PLOTS_DIR, "self_correction_waterfall.png"))
    
    plot_phase_portrait(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "phase_portrait_baseline.png"))
    plot_transition_heatmap(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "transition_heatmap_baseline.png"))
    quiver_transition_plot(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "trajectory_field_baseline.png"))
    plot_strip_transitions(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "strip_transitions_baseline.png"))
    plot_response_drift(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "response_drift_baseline.png"))
    plot_basin_map(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "basin_atlas_baseline.png"))
    plot_basin_heatmaps(df[df["variant"] == BASELINE_VARIANT], "baseline")

    for variant in [BASELINE_VARIANT, best_style]:
        sdf = df[df["variant"] == variant]
        plot_reliability(sdf, variant, "pass1", os.path.join(PLOTS_DIR, f"reliability_{variant}_pass1.png"))
        plot_reliability(sdf, variant, "pass2", os.path.join(PLOTS_DIR, f"reliability_{variant}_pass2.png"))

    print("Generating atlas maps...")
    plot_mean_trajectory(df, os.path.join(PLOTS_DIR, "trajectory_field_all_variants.png"))
    plot_strip_transitions(df, os.path.join(PLOTS_DIR, "strip_transitions_all_variants.png"))
    plot_response_drift(df, os.path.join(PLOTS_DIR, "response_drift_all_variants.png"))
    plot_bifurcation(df, os.path.join(PLOTS_DIR, "bifurcation_all_variants.png"))
    plot_basin_map(df, os.path.join(PLOTS_DIR, "basin_atlas_all_variants.png"))
    plot_basin_heatmaps(df, "all_variants")

    if PLOTLY_AVAILABLE:
        print("Generating Sankey diagrams...")
        for variant in [BASELINE_VARIANT, best_style]:
            sdf = df[df["variant"] == variant].copy()
            source = []
            target = []
            value = []
            nodes = ["W→W", "W→C", "C→W", "C→C"]
            trans_map = {
                "wrong→wrong": "W→W",
                "wrong→correct": "W→C",
                "correct→wrong": "C→W",
                "correct→correct": "C→C"
            }
            idx = {n: i for i, n in enumerate(nodes)}
            links = Counter(sdf["transition"].tolist())
            for s in ["wrong→wrong", "wrong→correct", "correct→wrong", "correct→correct"]:
                source.append(0)
                target.append(idx[trans_map[s]])
                value.append(int(links.get(s, 0)))
            fig = go.Figure(go.Sankey(
                arrangement="snap",
                node=dict(pad=18, thickness=20, label=["Start"] + nodes, color=["rgba(31,119,180,0.7)"] + ["rgba(100,100,100,0.65)"] * 4),
                link=dict(source=source, target=[t + 1 for t in target], value=value, color=["rgba(120,120,120,0.25)"] * len(value)),
            ))
            fig.update_layout(title_text=f"Self-correction transition Sankey ({variant})", font=dict(size=12), width=1200, height=700)
            try:
                fig.write_html(os.path.join(HTML_DIR, f"sankey_{variant}.html"))
            except Exception:
                pass
            try:
                fig.write_image(os.path.join(PLOTS_DIR, f"sankey_{variant}.png"), scale=2)
            except Exception:
                pass

    print("Generating question summaries...")
    plot_representative_trajectories(df, question_summary, os.path.join(PLOTS_DIR, "representative_trajectories_all.png"))
    plot_question_glyph_atlas(df, question_summary, os.path.join(PLOTS_DIR, "question_glyph_atlas_all.png"), top_n=12)

    q = question_summary.copy().sort_values(["stability_score", "transition_entropy"], ascending=[True, False])
    bar_plot(q["question_id"].astype(str).tolist(), q["stability_score"].tolist(), "Question stability scores", os.path.join(PLOTS_DIR, "question_stability_scores.png"), ylabel="Stability", rotate=90)
    histogram(q["transition_entropy"].tolist(), "Question transition entropy distribution", os.path.join(PLOTS_DIR, "question_transition_entropy_hist.png"), xlabel="Entropy")
    histogram(q["response_drift_mean"].tolist(), "Question response drift distribution", os.path.join(PLOTS_DIR, "question_response_drift_hist.png"), xlabel="Drift")
    scatter(q["stability_score"].tolist(), q["pass2_accuracy"].tolist(), "Stability vs pass-2 accuracy", os.path.join(PLOTS_DIR, "stability_vs_pass2_accuracy.png"), xlabel="Stability", ylabel="Pass-2 accuracy")
    scatter(q["transition_entropy"].tolist(), q["improvement_rate"].tolist(), "Transition entropy vs improvement rate", os.path.join(PLOTS_DIR, "entropy_vs_improvement.png"), xlabel="Transition entropy", ylabel="Improvement rate")
    scatter(q["response_drift_mean"].tolist(), q["delta_logodds_mean"].tolist(), "Response drift vs Δ log-odds", os.path.join(PLOTS_DIR, "drift_vs_delta_logodds.png"), xlabel="Response drift", ylabel="Δ log-odds")
    bar_plot(basin_counts["basin_class"].tolist(), basin_counts["count"].tolist(), "Question basin class counts", os.path.join(PLOTS_DIR, "basin_class_counts.png"), ylabel="Count", rotate=20)

    print("Generating aggregate dashboards...")
    fig, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)
    axes[0, 0].bar(variant_summary["variant"], variant_summary["pass1_raw_accuracy"], alpha=0.85)
    axes[0, 0].set_title("Pass-1 raw accuracy by variant")
    axes[0, 0].tick_params(axis="x", rotation=30)
    annotated_bar(axes[0, 0])

    axes[0, 1].bar(variant_summary["variant"], variant_summary["pass2_raw_accuracy"], alpha=0.85)
    axes[0, 1].set_title("Pass-2 raw accuracy by variant")
    axes[0, 1].tick_params(axis="x", rotation=30)
    annotated_bar(axes[0, 1])

    axes[1, 0].bar(variant_summary["variant"], variant_summary["improvement_rate"], alpha=0.85)
    axes[1, 0].set_title("Improvement rate by variant")
    axes[1, 0].tick_params(axis="x", rotation=30)
    annotated_bar(axes[1, 0])

    axes[1, 1].bar(variant_summary["variant"], variant_summary["regression_rate"], alpha=0.85)
    axes[1, 1].set_title("Regression rate by variant")
    axes[1, 1].tick_params(axis="x", rotation=30)
    annotated_bar(axes[1, 1])

    plt.suptitle("Self-correction basin atlas dashboard", fontsize=16, y=1.01)
    plt.savefig(os.path.join(PLOTS_DIR, "overall_dashboard.png"), dpi=180, bbox_inches="tight")
    plt.close()

    variant_metric = variant_summary.set_index("variant")[[
        "pass1_raw_accuracy", "pass2_raw_accuracy", "pass1_cal_accuracy", "pass2_cal_accuracy",
        "pass1_commitment_mean", "pass2_commitment_mean", "improvement_rate", "regression_rate",
        "transition_entropy", "q_stability_mean", "q_drift_mean"
    ]]
    heatmap(variant_metric.values, variant_metric.columns.tolist(), variant_metric.index.tolist(), "Variant metric heatmap", os.path.join(PLOTS_DIR, "variant_metric_heatmap.png"), xlabel="Metric", ylabel="Variant", cmap="coolwarm")
    
    # ---------------------------------------------------------
    # GENERATE FINAL MARKDOWN REPORT & JSON SUMMARY
    # ---------------------------------------------------------
    print("Writing final Markdown & JSON reports...")
    
    md = ["# StrategyQA self-correction basin atlas report\n"]
    md.append(f"- Model: `microsoft/phi-3-mini-4k-instruct`")
    md.append(f"- Validation samples: 120")
    md.append(f"- Test samples: 120")
    md.append(f"- Prompt variants: plain, anchored, anchored_reason, anchored_xml, anchored_selfcheck, answer_first, verbose")
    md.append(f"- Baseline variant: `plain`")
    md.append(f"- Deterministic decoding: True")
    md.append("\n## Overall summary\n")
    md.append("| Metric | Value |\n|---|---:|")
    for k, v in overall.items():
        if isinstance(v, (int, float)):
            md.append(f"| {k} | {v:.3f} |")
        else:
            md.append(f"| {k} | {v} |")

    md.append("\n## Variant summary\n")
    md.append("| Variant | Pass1 acc | Pass2 acc | Pass1 cal acc | Pass2 cal acc | Improve | Regress | Transition entropy | Drift |")
    md.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, r in variant_summary.sort_values("strength").iterrows():
        md.append(
            f"| {r['variant']} | {r['pass1_raw_accuracy']:.3f} | {r['pass2_raw_accuracy']:.3f} | {r['pass1_cal_accuracy']:.3f} | {r['pass2_cal_accuracy']:.3f} | {r['improvement_rate']:.3f} | {r['regression_rate']:.3f} | {r['transition_entropy']:.3f} | {r['mean_response_drift']:.3f} |"
        )

    md.append("\n## Question basin classes\n")
    md.append("| Basin class | Count |\n|---|---:|")
    for _, r in basin_counts.iterrows():
        md.append(f"| {r['basin_class']} | {int(r['count'])} |")

    md.append("\n## Interpretation\n")
    md.append("- A high stability score means the question rarely flips across prompt variants and passes.")
    md.append("- A high transition entropy means the sample moves through many different states (e.g. wrong→correct, correct→wrong).")
    md.append("- The phase portraits show whether pass2 sharpens or destabilizes the decision basin.")
    md.append("- The quiver map shows the direction of self-correction in confidence-commitment space.")
    md.append("- The improvement and regression basins reveal where self-correction is genuinely helpful and where it over-corrects.")
    md.append("- The response-drift plots show whether explanation drift aligns with confidence changes.")
    md.append("- The glyph atlas makes individual question behavior visually inspectable at a glance.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    # Reconstruct the summary.json configuration dictionary defaults
    config = {
        "BASE_MODEL": "microsoft/phi-3-mini-4k-instruct",
        "SFT_PATH": "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora",
        "N_VALIDATION": 120,
        "N_TEST": 120,
        "BASELINE_VARIANT": "plain",
        "PASS1_MAX_NEW": 48,
        "PASS2_MAX_NEW": 48,
        "TEMPERATURE": 0.0,
        "TOP_P": 0.9,
    }
    
    with open(os.path.join(REPORTS_DIR, "summary.json"), "w") as f:
        json.dump({
            "config": config,
            "overall": overall,
            "variant_summary": variant_summary.to_dict(orient="records"),
            "question_summary": question_summary.to_dict(orient="records"),
        }, f, indent=2)

    print(f"\n✅ All missing files have been generated.")
    print(f"✅ The complete output is fully restored at: {OUT_DIR}")

if __name__ == "__main__":
    main()

Copying saved data from /kaggle/input/datasets/adithyaled24b039/phi3-stratqa-self-correction-basin-atlas-csv/strategyqa_self_correction_basin_atlas to /kaggle/working/strategyqa_self_correction_basin_atlas...
Loading data from restored Output Directory...
Computing on-the-fly values & writing missing late-stage data...
Generating Grouped Bars & Violins...
Generating baseline & best-style dashboards...
Generating atlas maps...
Generating Sankey diagrams...
Generating question summaries...
Generating aggregate dashboards...
Writing final Markdown & JSON reports...

✅ All missing files have been generated.
✅ The complete output is fully restored at: /kaggle/working/strategyqa_self_correction_basin_atlas
